# ============================================================
# MOVIELENS 1M HYBRID MOVIE RECOMMENDATION SYSTEM
# User-Based Collaborative Filtering (User-CF) + Item-Based Collaborative Filtering (Item-CF)
# ============================================================

In [1]:
# ============================================================
# 1. IMPORTS & MOVIE DATA LOADING
# ============================================================

import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

MOVIES_PATH = r"D:\ADA_Netflix\Neflix-Recommendation\movies.dat"  # change path if required

movies = pd.read_csv(
    MOVIES_PATH,
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["MovieID", "Title", "Genres"]
)

print(f"Loaded {len(movies):,} movies.")


Loaded 3,883 movies.


In [2]:
# ============================================================
# 2. USER-BASED CF FEATURE EXTRACTION & COSINE SIMILARITY
#    18 genre features + normalized release year (19D representation)
# ============================================================

# Extract release year
movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)").astype(float)

# Remove year from title for cleaner display/search
movies["CleanTitle"] = (
    movies["Title"]
    .str.replace(r"\s*\(\d{4}\)", "", regex=True)
    .str.strip()
)

# 18 genre one-hot features
movies["GenreList"] = movies["Genres"].apply(
    lambda x: x.split("|") if x != "(no genres listed)" else []
)

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies["GenreList"])

# Normalize release year
year_values = movies["Year"].fillna(movies["Year"].median()).values.reshape(-1, 1)
scaler = MinMaxScaler()
year_feature = scaler.fit_transform(year_values)

# Combine genres + normalized year (19-dimensional feature vectors)
user_cf_features = np.hstack([
    genre_matrix,
    year_feature
])

# User-Based CF cosine similarity matrix (3883 x 3883)
user_cf_similarity = cosine_similarity(user_cf_features)

print(f"User-CF feature matrix shape: {user_cf_features.shape}")
print("User-Based CF similarity matrix created.")


User-CF feature matrix shape: (3883, 19)
User-Based CF similarity matrix created.


In [3]:
# ============================================================
# 3. RATINGS DATA LOADING & ITEM-BASED CF SIMILARITY
#    Item-User matrix -> Item-Item Cosine Similarity
# ============================================================

RATINGS_PATH = r"D:\ADA_Netflix\Neflix-Recommendation\ratings.dat"

ratings = pd.read_csv(
    RATINGS_PATH,
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["UserID", "MovieID", "Rating", "Timestamp"]
)

print(f"Loaded {len(ratings):,} ratings across {ratings['UserID'].nunique():,} users.")

# Build aligned Item-User rating matrix
movie_id_to_idx = {mid: i for i, mid in enumerate(movies["MovieID"])}
user_ids = ratings["UserID"].unique()
user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}

valid_ratings = ratings[ratings["MovieID"].isin(movie_id_to_idx)]
rows = valid_ratings["MovieID"].map(movie_id_to_idx).values
cols = valid_ratings["UserID"].map(user_id_to_idx).values
data = valid_ratings["Rating"].values

item_user_matrix = csr_matrix((data, (rows, cols)), shape=(len(movies), len(user_ids)))

# Item-CF item-item cosine similarity matrix (3883 x 3883)
item_cf_similarity = cosine_similarity(item_user_matrix, dense_output=True)

print(f"Item-User rating matrix shape: {item_user_matrix.shape}")
print("Item-Based CF similarity matrix created.")


Loaded 1,000,209 ratings across 6,040 users.
Item-User rating matrix shape: (3883, 6040)
Item-Based CF similarity matrix created.


In [4]:
# ============================================================
# 4. IMPROVED MOVIE SEARCH & SELECTION
#    Handles: Exact match, Single partial, Multiple matches, No match
# ============================================================

def find_movie(query_text):
    """
    Find the closest matching movie title with interactive selection for ambiguous queries.
    - Case A: Exact match -> Select directly
    - Case B: Single partial match -> Select automatically
    - Case C: Multiple matches -> Numbered interactive prompt
    - Case D: No match -> Return None with sample titles
    """
    if not query_text or not isinstance(query_text, str):
        return None
        
    query = query_text.lower().strip()
    if not query:
        return None

    # CASE A: Exact match (check CleanTitle or Title)
    exact = movies[
        (movies["CleanTitle"].str.lower() == query) |
        (movies["Title"].str.lower() == query)
    ]
    if len(exact) == 1:
        return exact.iloc[0]
    elif len(exact) > 1:
        matches = exact
    else:
        # Partial substring match (using regex escape)
        escaped_query = re.escape(query)
        matches = movies[
            movies["CleanTitle"].str.lower().str.contains(escaped_query, na=False) |
            movies["Title"].str.lower().str.contains(escaped_query, na=False)
        ]

    # CASE D: No match
    if len(matches) == 0:
        print(f"\nMovie not found.")
        print("\nTry a title such as:")
        sample_titles = movies["CleanTitle"].sample(5, random_state=42).tolist()
        for s in sample_titles:
            print(f"  - {s}")
        return None

    # CASE B: One partial match
    if len(matches) == 1:
        return matches.iloc[0]

    # CASE C: Multiple matches
    print(f"\nMultiple movies found:")
    for idx, (_, row) in enumerate(matches.iterrows(), start=1):
        year_str = f"({int(row['Year'])})" if not pd.isna(row['Year']) else ""
        print(f"\n{idx}. {row['CleanTitle']} {year_str}")
        print(f"   Genres: {row['Genres']}")

    print()
    while True:
        try:
            sel = input("Select a movie number: ").strip()
            choice = int(sel)
            if 1 <= choice <= len(matches):
                return matches.iloc[choice - 1]
            else:
                print(f"Please enter a number between 1 and {len(matches)}.")
        except (ValueError, EOFError):
            print("Invalid input. Please enter a valid number.")
            return None


### Hybrid Recommendation Formula

$$\text{Hybrid Score} = 0.5 \times \text{User-Based CF Similarity} + 0.5 \times \text{Item-Based CF Similarity}$$


In [5]:
# ============================================================
# 5. RECOMMENDATION EXPLANATION FUNCTION
# ============================================================

def explain_recommendation(input_movie, rec_movie, user_cf_norm_score, item_cf_norm_score):
    """
    Generates a deterministic explanation for why a movie was recommended.
    """
    input_genres = set(input_movie["GenreList"])
    rec_genres = set(rec_movie["GenreList"])
    shared = input_genres.intersection(rec_genres)
    
    if shared:
        shared_str = ", ".join(sorted(shared))
    else:
        shared_str = "None"
        
    if not pd.isna(input_movie["Year"]) and not pd.isna(rec_movie["Year"]):
        diff = int(abs(input_movie["Year"] - rec_movie["Year"]))
        year_str = f"{diff} year{'s' if diff != 1 else ''}"
    else:
        year_str = "N/A"
        
    diff_signal = user_cf_norm_score - item_cf_norm_score
    if abs(diff_signal) <= 0.05:
        signal_str = "Both User-CF and Item-CF collaborative signals contribute strongly."
    elif diff_signal > 0.05:
        signal_str = "User-based collaborative signal is the stronger signal."
    else:
        signal_str = "Item-based collaborative signal is the stronger signal."
        
    explanation = (
        f"   Why recommended:\n"
        f"   • Shared genres: {shared_str}\n"
        f"   • Year difference: {year_str}\n"
        f"   • User-CF similarity: {user_cf_norm_score:.3f}\n"
        f"   • Item-CF similarity: {item_cf_norm_score:.3f}\n"
        f"   • {signal_str}"
    )
    return explanation


# ============================================================
# 6. HYBRID RECOMMENDATION FUNCTION (User-CF + Item-CF)
# ============================================================

def normalize_scores(arr):
    min_val = np.min(arr)
    max_val = np.max(arr)
    if max_val - min_val > 0:
        return (arr - min_val) / (max_val - min_val)
    return np.zeros_like(arr)


def recommend_movies(movie_name, n=10):
    """
    Recommends Top-N movies using Hybrid Filtering (User-Based CF + Item-Based CF).
    """
    movie = find_movie(movie_name)
    if movie is None:
        return None

    movie_index = movie.name

    # 1. User-Based CF similarity scores
    user_cf_scores = user_cf_similarity[movie_index]

    # 2. Item-Based CF similarity scores
    item_cf_scores = item_cf_similarity[movie_index]

    # 3. Normalize scores to [0, 1]
    user_cf_norm = normalize_scores(user_cf_scores)
    item_cf_norm = normalize_scores(item_cf_scores)

    # 4. Hybrid score (equal weights: 0.5 * User-CF + 0.5 * Item-CF)
    hybrid_scores = 0.5 * user_cf_norm + 0.5 * item_cf_norm

    # 5. Sort descending and exclude the input movie itself
    similar_indices = np.argsort(hybrid_scores)[::-1]
    similar_indices = [i for i in similar_indices if i != movie_index]

    # 6. Top N recommendations
    top_indices = similar_indices[:n]
    recommendations = movies.iloc[top_indices].copy()
    
    recommendations["HybridScore"] = hybrid_scores[top_indices]
    recommendations["UserCFScore"] = user_cf_scores[top_indices]
    recommendations["ItemCFScore"] = item_cf_scores[top_indices]
    recommendations["UserCFNorm"] = user_cf_norm[top_indices]
    recommendations["ItemCFNorm"] = item_cf_norm[top_indices]

    print("\n" + "=" * 70)
    print("MOVIELENS 1M HYBRID MOVIE RECOMMENDER")
    print("=" * 70)
    year_display = f"({int(movie['Year'])})" if not pd.isna(movie['Year']) else ""
    print(f"\nSelected Movie: {movie['CleanTitle']} {year_display}")
    print(f"Genres: {movie['Genres']}")
    print("\nHybrid Method:")
    print("0.5 × User-Based CF Similarity + 0.5 × Item-Based CF Similarity")
    print("\n" + "=" * 70)
    print("TOP 10 HYBRID RECOMMENDATIONS")
    print("=" * 70 + "\n")

    for rank, (_, rec) in enumerate(recommendations.iterrows(), start=1):
        rec_year = f"({int(rec['Year'])})" if not pd.isna(rec['Year']) else ""
        print(f"{rank:2}. {rec['CleanTitle']} {rec_year}")
        print(f"   Genres: {rec['Genres']}")
        print(f"   Hybrid Score: {rec['HybridScore']:.3f} | User-CF Similarity: {rec['UserCFScore']:.3f} | Item-CF Similarity: {rec['ItemCFScore']:.3f}")
        exp = explain_recommendation(movie, rec, rec["UserCFNorm"], rec["ItemCFNorm"])
        print(exp + "\n")

    return recommendations[["MovieID", "CleanTitle", "Year", "Genres", "HybridScore", "UserCFScore", "ItemCFScore"]]


In [6]:
# ============================================================
# 7. INTERACTIVE DEMO
# ============================================================

movie_name = input("\nEnter a movie name: ")

recommend_movies(movie_name)


Enter a movie name: Godfather
Multiple movies found:
1. Godfather, The (1972)
   Genres: Action|Crime|Drama
2. Godfather: Part II, The (1974)
   Genres: Action|Crime|Drama
3. Godfather: Part III, The (1990)
   Genres: Action|Crime|Drama

Select a movie number: 1
MOVIELENS 1M HYBRID MOVIE RECOMMENDER

Selected Movie: Godfather, The (1972)
Genres: Action|Crime|Drama

Hybrid Method:
0.5 × User-Based CF Similarity + 0.5 × Item-Based CF Similarity

TOP 10 HYBRID RECOMMENDATIONS

 1. Godfather: Part II, The (1974)
   Genres: Action|Crime|Drama
   Hybrid Score: 0.885 | User-CF Similarity: 1.000 | Item-CF Similarity: 0.770
   Why recommended:
   • Shared genres: Action, Crime, Drama
   • Year difference: 2 years
   • User-CF similarity: 1.000
   • Item-CF similarity: 0.770
   • User-based collaborative signal is the stronger signal.

 2. Untouchables, The (1987)
   Genres: Action|Crime|Drama
   Hybrid Score: 0.757 | User-CF Similarity: 0.996 | Item-CF Similarity: 0.518
   Why recommended:
   •